# Credit Risk Grading System
**End-to-end pipeline:** SEC EDGAR financials - FinBERT sentiment (Item 1A) - OpenRouter LLM scoring (full MD&A) - Calibrated LightGBM PD model - LGD - Expected Loss - Stress Testing - SHAP - Dashboard

**Data sources:** UCLA LoPucki BRD (defaulted) + S&P 500 (healthy) | Year-matched universe

**Regulatory features:** Time-based train/test split | CalibratedClassifierCV (isotonic) | SHAP explainability

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
!pip install -q groq transformers torch lightgbm plotly shap scikit-learn \
             fredapi sec-edgar-downloader beautifulsoup4 openpyxl tiktoken requests

import os, json, time, re, random, threading, warnings, itertools
import requests
import pandas as pd
import numpy as np
from google.colab import drive
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch
import tiktoken

warnings.filterwarnings('ignore')
random.seed(42)
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/credit_risk_v3'
DATA_DIR    = f'{BASE_DIR}/data'
EDGAR_CACHE = f'{BASE_DIR}/cache/edgar'
TENK_CACHE  = f'{BASE_DIR}/cache/tenk'

for d in [DATA_DIR, EDGAR_CACHE, TENK_CACHE]:
    os.makedirs(d, exist_ok=True)

HEADERS = {'User-Agent': 'varun your_email@gmail.com'}

# Tiktoken encoder for token counting
enc = tiktoken.get_encoding('cl100k_base')

print(f'GPU available : {torch.cuda.is_available()}')
print(f'Device        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Base dir      : {BASE_DIR}')
print('Setup complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.3 MB/s eta 0:00:00
Mounted at /content/drive
GPU available : True
Device        : Tesla T4
Base dir      : /content/drive/MyDrive/credit_risk_v3
Setup complete.


In [ ]:
# ── Cell 2: Universe (100 Defaulted + 100 Healthy, Matched Years) ─────────────
import io
from google.colab import files

LOPUCKI_PATH = f'{DATA_DIR}/lopucki.xlsx'
if not os.path.exists(LOPUCKI_PATH):
    print('Upload LoPucki xlsx...')
    uploaded = files.upload()
    import shutil
    shutil.copy(list(uploaded.keys())[0], LOPUCKI_PATH)
    print('Saved to Drive.')

lopucki = pd.read_excel(LOPUCKI_PATH)
lopucki['Chapter']      = lopucki['Chapter'].astype(str).str.strip()
lopucki['AssetsBefore'] = pd.to_numeric(lopucki['AssetsBefore'], errors='coerce')
lopucki['YearFiled']    = pd.to_numeric(lopucki['YearFiled'],    errors='coerce')
lopucki['CikBefore']    = pd.to_numeric(lopucki['CikBefore'],    errors='coerce')

bankrupt_raw = lopucki[
    (lopucki['Chapter']      == '11') &
    (lopucki['YearFiled']    >= 2010) &
    (lopucki['AssetsBefore'] >= 100)  &
    (lopucki['CikBefore'].notna())
].copy()

bankrupt_raw['cik']         = bankrupt_raw['CikBefore'].apply(lambda x: str(int(x)).zfill(10))
bankrupt_raw                = bankrupt_raw[['NameCorp','cik','YearFiled','AssetsBefore','SICDescription']].copy()
bankrupt_raw.columns        = ['name','cik','default_year','assets_before','industry']
bankrupt_raw['defaulted']   = 1
bankrupt_raw['target_year'] = (bankrupt_raw['default_year'] - 2).astype(int)
bankrupt_raw                = bankrupt_raw.drop_duplicates(subset='cik')
print(f'LoPucki filtered: {len(bankrupt_raw)} defaulted companies')

sp500_url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv'
sp500     = pd.read_csv(io.StringIO(requests.get(sp500_url).text))
sp500['CIK'] = pd.to_numeric(sp500['CIK'], errors='coerce')
sp500        = sp500[sp500['CIK'].notna()].copy()
sp500['cik'] = sp500['CIK'].astype(int).astype(str).str.zfill(10)
sp500        = sp500[~sp500['cik'].isin(set(bankrupt_raw['cik']))].copy()
healthy_df             = sp500[['Security','cik','GICS Sector']].copy()
healthy_df.columns     = ['name','cik','industry']
healthy_df['defaulted']= 0
healthy_df['default_year'] = None
print(f'S&P 500 healthy: {len(healthy_df)} companies')

sampled_def = bankrupt_raw.groupby('industry', group_keys=False).apply(
    lambda x: x.sample(min(len(x), max(1, int(100*len(x)/len(bankrupt_raw)))), random_state=42)
).head(100)
if len(sampled_def) < 100:
    remaining   = bankrupt_raw[~bankrupt_raw['cik'].isin(sampled_def['cik'])]
    sampled_def = pd.concat([sampled_def, remaining.sample(100-len(sampled_def), random_state=42)])

sampled_healthy = healthy_df.sample(100, random_state=42).copy()
year_dist       = sampled_def['target_year'].value_counts().to_dict()
year_pool       = []
for yr, cnt in year_dist.items():
    year_pool.extend([yr]*cnt)
while len(year_pool) < 100:
    year_pool.append(random.choice(list(year_dist.keys())))
random.shuffle(year_pool)
sampled_healthy['target_year'] = year_pool[:100]
sampled_healthy['target_year'] = sampled_healthy['target_year'].astype(int)

KEEP = ['cik','name','target_year','defaulted','industry']
universe = pd.concat([
    sampled_def[KEEP],
    sampled_healthy[KEEP]
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Final universe: {len(universe)} | Defaulted: {universe["defaulted"].sum()} | Healthy: {(universe["defaulted"]==0).sum()}')
print(f'Year distribution:')
print(universe.groupby(['defaulted','target_year']).size().unstack(fill_value=0))
universe.to_csv(f'{DATA_DIR}/universe_200.csv', index=False)
print('Saved universe_200.csv')

LoPucki filtered: 308 defaulted companies
S&P 500 healthy: 501 companies
Final universe: 200 | Defaulted: 100 | Healthy: 100
Year distribution:
target_year  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  \
defaulted                                                                       
0              12     6     8     6     5    10    15     8     5     7    13   
1              12     6     8     6     5    10    15     8     5     7    13   

target_year  2020  
defaulted          
0               5  
1               5  
Saved universe_200.csv


In [ ]:
# ── Cell 3: EDGAR Financial Pull ──────────────────────────────────────────────
CONCEPTS = {
    'Assets':             ['Assets'],
    'Liabilities':        ['Liabilities'],
    'LongTermDebt':       ['LongTermDebt','LongTermDebtNoncurrent','LongTermNotesPayable'],
    'StockholdersEquity': ['StockholdersEquity',
                           'StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest'],
    'NetIncome':          ['NetIncomeLoss','ProfitLoss'],
    'OperatingIncome':    ['OperatingIncomeLoss'],
    'InterestExpense':    ['InterestAndDebtExpense','InterestExpense','InterestExpenseDebt'],
    'AssetsCurrent':      ['AssetsCurrent'],
    'LiabilitiesCurrent': ['LiabilitiesCurrent'],
    'OperatingCashFlow':  ['NetCashProvidedByUsedInOperatingActivities'],
    'Revenue':            ['Revenues','RevenueFromContractWithCustomerExcludingAssessedTax','SalesRevenueNet'],
}

def fetch_facts(cik):
    path = f'{EDGAR_CACHE}/facts_{cik}.json'
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    url  = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        facts = resp.json()
        with open(path, 'w') as f:
            json.dump(facts, f)
        return facts
    except:
        return None

def get_metric(facts, concepts, target_year):
    us_gaap = facts.get('facts', {}).get('us-gaap', {})
    for concept in concepts:
        if concept not in us_gaap:
            continue
        usd = [x for x in us_gaap[concept].get('units',{}).get('USD',[])
               if x.get('form') == '10-K' and x.get('end','')[:4].isdigit()]
        if not usd:
            continue
        best = sorted(usd, key=lambda x: abs(int(x['end'][:4]) - target_year))[0]
        if abs(int(best['end'][:4]) - target_year) <= 1:
            return best['val']
    return None

universe  = pd.read_csv(f'{DATA_DIR}/universe_200.csv')
universe['cik'] = universe['cik'].astype(str).str.zfill(10)

rows, errors, skipped = [], 0, 0
PARTIAL   = f'{DATA_DIR}/edgar_partial.csv'
done_ciks = set()

if os.path.exists(PARTIAL):
    part      = pd.read_csv(PARTIAL)
    rows      = part.to_dict('records')
    done_ciks = set(part['cik'].astype(str).str.zfill(10))
    print(f'Resuming: {len(done_ciks)} done')

for i, row in universe.iterrows():
    cik  = str(row['cik']).zfill(10)
    year = int(row['target_year'])
    if cik in done_ciks:
        continue
    facts = fetch_facts(cik)
    if facts is None:
        errors += 1
        done_ciks.add(cik)
        continue
    data = {'cik': cik, 'name': row['name'], 'target_year': year,
            'defaulted': int(row['defaulted']), 'industry': row['industry']}
    for metric, concepts in CONCEPTS.items():
        data[metric] = get_metric(facts, concepts, year)
    if all(data[m] is None for m in CONCEPTS):
        skipped += 1
        done_ciks.add(cik)
        continue
    rows.append(data)
    done_ciks.add(cik)
    if (i+1) % 25 == 0:
        print(f'[{i+1}/200] collected={len(rows)} errors={errors} skipped={skipped}')
        pd.DataFrame(rows).to_csv(PARTIAL, index=False)
    time.sleep(0.05)

edgar_df = pd.DataFrame(rows)
edgar_df.to_csv(f'{DATA_DIR}/edgar_raw.csv', index=False)
if os.path.exists(PARTIAL):
    os.remove(PARTIAL)
print(f'Done. Collected={len(edgar_df)} Errors={errors} Skipped={skipped}')
print(f'Defaulted={edgar_df["defaulted"].sum()} Healthy={(edgar_df["defaulted"]==0).sum()}')
print(f'Missing:\n{edgar_df[list(CONCEPTS.keys())].isnull().sum()}')

[25/200] collected=21 errors=0 skipped=4
[50/200] collected=46 errors=0 skipped=4
[100/200] collected=89 errors=3 skipped=8
[125/200] collected=112 errors=4 skipped=9
[150/200] collected=132 errors=7 skipped=11
[175/200] collected=153 errors=11 skipped=11
[200/200] collected=175 errors=11 skipped=14
Done. Collected=175 Errors=11 Skipped=14
Defaulted=82 Healthy=93
Missing:
Assets                 6
Liabilities           55
LongTermDebt          46
StockholdersEquity     5
NetIncome              5
OperatingIncome       40
InterestExpense       27
AssetsCurrent         31
LiabilitiesCurrent    31
OperatingCashFlow     18
Revenue               21
dtype: int64


In [ ]:
# ── Cell 4: Ratio Engineering + Altman Z + Ohlson O ──────────────────────────
def safe_div(a, b):
    return (pd.to_numeric(a, errors='coerce') /
            pd.to_numeric(b, errors='coerce')).replace([np.inf,-np.inf], np.nan)

df = pd.read_csv(f'{DATA_DIR}/edgar_raw.csv')
print(f'Loaded: {df.shape}')

mask = df['Liabilities'].isna() & df['Assets'].notna() & df['StockholdersEquity'].notna()
df.loc[mask, 'Liabilities'] = df.loc[mask,'Assets'] - df.loc[mask,'StockholdersEquity']
print(f'Liabilities imputed: {mask.sum()} rows')

df['leverage_ratio']    = safe_div(df['Liabilities'],       df['Assets'])
df['debt_to_equity']    = safe_div(df['LongTermDebt'],      df['StockholdersEquity'])
df['debt_to_assets']    = safe_div(df['LongTermDebt'],      df['Assets'])
df['current_ratio']     = safe_div(df['AssetsCurrent'],     df['LiabilitiesCurrent'])
df['working_capital']   = (pd.to_numeric(df['AssetsCurrent'], errors='coerce') -
                           pd.to_numeric(df['LiabilitiesCurrent'], errors='coerce'))
df['roa']               = safe_div(df['NetIncome'],         df['Assets'])
df['net_profit_margin'] = safe_div(df['NetIncome'],         df['Revenue'])
df['asset_turnover']    = safe_div(df['Revenue'],           df['Assets'])
df['interest_coverage'] = safe_div(df['OperatingIncome'],   df['InterestExpense'])
df['ocf_to_debt']       = safe_div(df['OperatingCashFlow'], df['LongTermDebt'])
df['ocf_to_assets']     = safe_div(df['OperatingCashFlow'], df['Assets'])
df['ocf_margin']        = safe_div(df['OperatingCashFlow'], df['Revenue'])
df['log_assets']        = np.log1p(pd.to_numeric(df['Assets'], errors='coerce').clip(lower=0))

CLIP = {
    'leverage_ratio':    (-0.5, 5.0), 'debt_to_equity':    (-50, 50),
    'debt_to_assets':    (0, 5.0),    'current_ratio':     (0, 20),
    'roa':               (-2, 2),     'net_profit_margin': (-10, 5),
    'asset_turnover':    (0, 10),     'interest_coverage': (-50, 50),
    'ocf_to_debt':       (-5, 5),     'ocf_to_assets':     (-2, 2),
    'ocf_margin':        (-5, 2),
}
for col, (lo, hi) in CLIP.items():
    df[col] = df[col].clip(lower=lo, upper=hi)

assets      = pd.to_numeric(df['Assets'],             errors='coerce')
liabilities = pd.to_numeric(df['Liabilities'],        errors='coerce')
equity      = pd.to_numeric(df['StockholdersEquity'], errors='coerce')
net_income  = pd.to_numeric(df['NetIncome'],          errors='coerce')
op_income   = pd.to_numeric(df['OperatingIncome'],    errors='coerce')
revenue     = pd.to_numeric(df['Revenue'],            errors='coerce')
curr_assets = pd.to_numeric(df['AssetsCurrent'],      errors='coerce')
curr_liab   = pd.to_numeric(df['LiabilitiesCurrent'], errors='coerce')

X1 = (curr_assets - curr_liab) / assets
X2 = net_income  / assets
X3 = op_income   / assets
X4 = equity      / liabilities.replace(0, np.nan)
X5 = revenue     / assets
df['altman_z']  = (1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5).clip(-10, 20)

def z_to_zone(z):
    if pd.isna(z):    return 'Unknown'
    elif z > 2.99:    return 'Safe'
    elif z > 1.81:    return 'Grey'
    else:             return 'Distress'
df['z_zone']    = df['altman_z'].apply(z_to_zone)
df['z_distress']= (df['z_zone'] == 'Distress').astype(int)
df['z_safe']    = (df['z_zone'] == 'Safe').astype(int)

SIZE  = np.log1p(assets.clip(lower=0))
TLTA  = liabilities / assets
WCTA  = (curr_assets - curr_liab) / assets
CLCA  = curr_liab / curr_assets.replace(0, np.nan)
OENEG = (liabilities > assets).astype(float)
NITA  = net_income / assets
FUTL  = op_income  / liabilities.replace(0, np.nan)
INTWO = (net_income < 0).astype(float)
CHIN  = net_income / (net_income.abs() + 1)
o_raw = (-1.32 - 0.407*SIZE + 6.03*TLTA - 1.43*WCTA + 0.076*CLCA
         - 1.72*OENEG - 2.37*NITA - 1.83*FUTL + 0.285*INTWO - 0.521*CHIN)
df['ohlson_o']    = o_raw.clip(-10, 10)
df['ohlson_prob'] = (1 / (1 + np.exp(-o_raw))).clip(0, 1)

RATIO_COLS = list(CLIP.keys()) + ['log_assets','working_capital','altman_z','ohlson_o','ohlson_prob','z_distress','z_safe']
META_COLS  = ['cik','name','target_year','defaulted','industry']
ratio_df   = df[META_COLS + RATIO_COLS].copy()
sparse     = ratio_df[RATIO_COLS].isnull().sum(axis=1) > len(RATIO_COLS) * 0.6
ratio_df   = ratio_df[~sparse].copy()

print(f'Ratio dataset: {ratio_df.shape}')
print(f'Defaulted={ratio_df["defaulted"].sum()} Healthy={(ratio_df["defaulted"]==0).sum()}')
print(f'Altman Z by default:\n{ratio_df.groupby("defaulted")["altman_z"].describe().round(3)}')
ratio_df.to_csv(f'{DATA_DIR}/ratio_dataset.csv', index=False)
print('Saved ratio_dataset.csv')

Loaded: (175, 16)
Liabilities imputed: 49 rows
Ratio dataset: (166, 23)
Defaulted=77 Healthy=89
Altman Z by default:
           count   mean    std    min    25%    50%    75%    max
defaulted                                                        
0           56.0  1.614  1.455  0.073  0.728  1.182  1.978  8.920
1           52.0  0.561  0.922 -2.893  0.148  0.550  1.012  2.195
Saved ratio_dataset.csv


In [ ]:
# ── Cell 5: 10-K Text Extraction (Item 1A: 1500w | Item 7: 5000w) ─────────────
def get_10k_doc_url(cik, target_year):
    cache_path = f'{TENK_CACHE}/submissions_{cik}.json'
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            subs = json.load(f)
    else:
        url  = f'https://data.sec.gov/submissions/CIK{cik}.json'
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        subs = resp.json()
        with open(cache_path, 'w') as f:
            json.dump(subs, f)
        time.sleep(0.12)

    filings   = subs.get('filings',{}).get('recent',{})
    forms     = filings.get('form',[])
    dates     = filings.get('filingDate',[])
    accnums   = filings.get('accessionNumber',[])
    prim_docs = filings.get('primaryDocument',[])

    candidates = []
    for form, date, acc, pdoc in zip(forms, dates, accnums, prim_docs):
        if form == '10-K' and date and pdoc:
            year = int(date[:4])
            if abs(year - target_year) <= 1:
                acc_clean = acc.replace('-','')
                doc_url   = f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_clean}/{pdoc}'
                candidates.append((abs(year - target_year), doc_url))
    if not candidates:
        return None
    return sorted(candidates)[0][1]

def find_nth(pattern, text, n=2):
    matches = list(re.finditer(pattern, text, re.IGNORECASE))
    if len(matches) >= n:
        return matches[n-1]
    return matches[-1] if matches else None

def extract_sections(html):
    """
    Extract Item 1A (1500 words for FinBERT) and Item 7 (5000 words for LLM).
    Uses 2nd regex match to skip Table of Contents.
    """
    clean = re.sub(r'<[^>]+>', ' ', html)
    clean = re.sub(r'&[a-zA-Z0-9#]+;', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    sections = []

    # Item 1A — Risk Factors (1500 words - FinBERT truncates to 250)
    ms_1a = find_nth(r'item\s+1a[\.|\s|:]+risk\s+factor', clean, n=2)
    me_1a = find_nth(r'item\s+1b[\.|\s|:]+', clean, n=2)
    if ms_1a:
        start = ms_1a.start()
        end   = me_1a.start() if (me_1a and me_1a.start() > start) else start + 15000
        words = clean[start:end].split()[:1500]
        if len(words) > 50:
            sections.append('[Item 1A]\n' + ' '.join(words))

    # Item 7 — MD&A (5000 words — full section for LLM)
    ms_7 = find_nth(r'item\s+7[\.|\s|:]+management', clean, n=2)
    me_7 = find_nth(r'item\s+7a[\.|\s|:]+', clean, n=2)
    if ms_7:
        start = ms_7.start()
        end   = me_7.start() if (me_7 and me_7.start() > start) else start + 50000
        words = clean[start:end].split()[:5000]
        if len(words) > 50:
            sections.append('[Item 7]\n' + ' '.join(words))

    if not sections:
        words = clean.split()
        mid   = (len(words) * 2) // 3
        sections.append('[Fallback]\n' + ' '.join(words[mid:mid+2000]))

    return '\n\n'.join(sections)

def fetch_10k_text(cik, target_year):
    path = f'{TENK_CACHE}/text_{cik}_{target_year}.txt'
    if os.path.exists(path):
        with open(path) as f:
            return f.read()
    doc_url = get_10k_doc_url(cik, target_year)
    if not doc_url:
        return None
    try:
        resp = requests.get(doc_url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            return None
        text = extract_sections(resp.text)
        if len(text.split()) < 100:
            return None
        with open(path, 'w', encoding='utf-8') as f:
            f.write(text)
        return text
    except:
        return None

ratio_df = pd.read_csv(f'{DATA_DIR}/ratio_dataset.csv')
ratio_df['cik'] = ratio_df['cik'].astype(str).str.zfill(10)
print(f'Loaded: {len(ratio_df)} companies')

test = fetch_10k_text('0000320193', 2022)
if test:
    item1a_w = len(test.split('[Item 7]')[0].split()) if '[Item 7]' in test else 0
    item7_w  = len(test.split('[Item 7]')[1].split()) if '[Item 7]' in test else 0
    print(f'Apple test OK: Item 1A={item1a_w}w | Item 7={item7_w}w | Total={len(test.split())}w')
else:
    print('Apple test FAILED')

Loaded: 166 companies
Apple test OK: Item 1A=1502w | Item 7=2802w | Total=4306w


In [ ]:
# ── Cell 6: FinBERT on Item 1A (Risk Factors) ─────────────────────────────────
# FinBERT scores Item 1A only — risk factors use explicit negative language
# Item 7 (MD&A) scored by OpenRouter in Cell 7 — management tone is always optimistic
from transformers import pipeline
import torch

print(f'Loading FinBERT on {"GPU" if torch.cuda.is_available() else "CPU"}...')
finbert = pipeline(
    'text-classification',
    model  = 'ProsusAI/finbert',
    device = 0 if torch.cuda.is_available() else -1,
    top_k  = None,
)
print('FinBERT loaded.')

FINBERT_CACHE = f'{DATA_DIR}/finbert_cache.json'
fb_cache      = {}
if os.path.exists(FINBERT_CACHE):
    with open(FINBERT_CACHE) as f:
        fb_cache = json.load(f)
    print(f'FinBERT cache: {len(fb_cache)} entries')

def score_section(t):
    """Score 250-word text with FinBERT. Returns pos/neg/neutral dict."""
    DEFAULT = {'positive': 0.333, 'negative': 0.333, 'neutral': 0.333}
    if not t or len(t.split()) < 10:
        return DEFAULT
    try:
        truncated = ' '.join(t.split()[:250])
        result    = finbert(truncated, truncation=True, max_length=512)[0]
        return {r['label']: round(r['score'], 6) for r in result}
    except Exception as e:
        return DEFAULT

def get_finbert_scores(cik, text):
    """Score Item 1A with FinBERT. Cached. Always returns dict."""
    cache_key = str(cik)
    if cache_key in fb_cache:
        return fb_cache[cache_key]

    # Extract Item 1A only
    item1a_text = ''
    if '[Item 1A]' in text:
        item1a_text = text.split('[Item 7]')[0].replace('[Item 1A]','').strip() \
                      if '[Item 7]' in text else text.replace('[Item 1A]','').strip()
    else:
        item1a_text = text

    scores = score_section(item1a_text)

    result = {
        '1a_negative': scores.get('negative', 0.333),
        '1a_positive': scores.get('positive', 0.333),
        '1a_neutral':  scores.get('neutral',  0.333),
    }
    fb_cache[cache_key] = result
    return result

ratio_df = pd.read_csv(f'{DATA_DIR}/ratio_dataset.csv')
ratio_df['cik'] = ratio_df['cik'].astype(str).str.zfill(10)
print(f'Loaded: {len(ratio_df)} companies')

fb_results = []
no_text    = 0

for i, row in ratio_df.iterrows():
    cik  = str(row['cik']).zfill(10)
    year = int(row['target_year'])
    text = fetch_10k_text(cik, year)
    if text is None:
        no_text += 1
        fb_results.append({'cik': cik, '1a_negative': np.nan,
                           '1a_positive': np.nan, '1a_neutral': np.nan,
                           'finbert_status': 'no_text'})
        continue
    scores = get_finbert_scores(cik, text)
    fb_results.append({'cik': cik, **scores, 'finbert_status': 'ok'})
    if (i+1) % 20 == 0:
        print(f'  [{i+1}/{len(ratio_df)}] processed | no_text={no_text}')
        with open(FINBERT_CACHE, 'w') as f:
            json.dump(fb_cache, f)

with open(FINBERT_CACHE, 'w') as f:
    json.dump(fb_cache, f)

finbert_df = pd.DataFrame(fb_results)
finbert_df['cik'] = finbert_df['cik'].astype(str).str.zfill(10)
finbert_df.to_csv(f'{DATA_DIR}/finbert_signals.csv', index=False)

print(f'Done. OK={(finbert_df["finbert_status"]=="ok").sum()} No_text={no_text}')
merged_check = ratio_df.merge(finbert_df, on='cik', how='left')
print(f'FinBERT 1A scores by default status:')
print(merged_check.groupby('defaulted')[['1a_negative','1a_positive']].mean().round(4))
print('Saved finbert_signals.csv')

Loading FinBERT on GPU...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT loaded.
FinBERT cache: 105 entries
Loaded: 166 companies
  [60/166] processed | no_text=19
  [80/166] processed | no_text=25
  [100/166] processed | no_text=31
  [140/166] processed | no_text=49
  [160/166] processed | no_text=57
Done. OK=105 No_text=61
FinBERT 1A scores by default status:
           1a_negative  1a_positive
defaulted                          
0               0.3895       0.0554
1               0.2894       0.0504
Saved finbert_signals.csv


In [ ]:
# ── Cell 7: OpenRouter LLM Numeric Scoring — Full MD&A (4500 tokens) ──────────
# Uses tiktoken for precise token counting — no 413 errors
# gpt-oss-120b with reasoning enabled for accurate credit scoring

OPENROUTER_KEY = "your-api-key"

SYSTEM_PROMPT = """You are a senior credit risk analyst. Score this 10-K MD&A section on 6 dimensions.

CRITICAL RULES:
- Use the FULL range 0-10. Do NOT default to 2 for everything.
- Be granular: use 3,4,5,6,7,8 not just 0,2,5,10.
- Higher score ALWAYS means MORE risk or distress.
- Base scores only on explicit evidence in the text.
- A company reporting operating losses, declining revenue, or debt concerns MUST score above 4.

SCORING RUBRIC:

going_concern_severity (0-10):
  2=standard boilerplate, 4=liquidity pressure mentioned,
  6=substantial doubt language, 8=auditor opinion, 10=imminent bankruptcy

debt_risk_score (0-10):
  2=manageable debt, 4=elevated leverage/covenant tightness,
  6=near covenant limits, 8=covenant breach/waiver, 10=default/restructuring

liquidity_risk_score (0-10):
  2=strong cash/credit lines, 4=adequate but pressured,
  6=reliant on credit lines, 8=lines drawn/cash burn, 10=unable to meet obligations

management_distress (0-10):
  2=highly confident/strong guidance, 4=positive with caution,
  6=neutral/mixed, 8=cautious/poor outlook, 10=highly distressed/no guidance

restructuring_risk (0-10):
  2=no mention, 4=cost cuts mentioned,
  6=significant restructuring, 8=major asset sales, 10=bankruptcy filing

revenue_distress (0-10):
  2=strong growth, 4=solid stable,
  6=flat/slight decline, 8=meaningful decline, 10=severe sustained decline

Return ONLY valid JSON:
{"going_concern_severity": int, "debt_risk_score": int, "liquidity_risk_score": int,
 "management_distress": int, "restructuring_risk": int, "revenue_distress": int}"""

USER_TEMPLATE = """Analyze this full MD&A section and return 6 numeric credit risk scores.
A company reporting losses or declining revenue MUST score above 4 on relevant dimensions.

MD&A text:
{text}

Return ONLY the JSON object with 6 integer scores."""

NULL_SIGNALS = {
    'going_concern_severity': np.nan, 'debt_risk_score': np.nan,
    'liquidity_risk_score': np.nan,   'management_distress': np.nan,
    'restructuring_risk': np.nan,     'revenue_distress': np.nan,
}

def truncate_to_tokens(text, max_tokens=4500):
    """Truncate text to exact token limit using tiktoken."""
    tokens = enc.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return enc.decode(tokens[:max_tokens])

def get_item7_text(cik, target_year):
    """Return full Item 7 text, truncated to 4500 tokens."""
    full_text = fetch_10k_text(cik, target_year)
    if full_text is None:
        return None
    item7 = full_text.split('[Item 7]')[1].strip() if '[Item 7]' in full_text else full_text.strip()
    return truncate_to_tokens(item7, max_tokens=4500)

def call_openrouter(text, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.post(
                url='https://openrouter.ai/api/v1/chat/completions',
                headers={
                    'Authorization': f'Bearer {OPENROUTER_KEY}',
                    'Content-Type': 'application/json',
                },
                data=json.dumps({
                    'model': 'openai/gpt-oss-120b:free',
                    'messages': [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': USER_TEMPLATE.format(text=text)}
                    ],
                    'reasoning': {'enabled': True}
                }),
                timeout=60,
            )
            if resp.status_code == 429:
                wait = int(resp.headers.get('Retry-After', 15))
                print(f'    Rate limited — waiting {wait}s')
                time.sleep(wait)
                continue
            if resp.status_code != 200:
                time.sleep(2**attempt)
                continue
            data    = resp.json()
            content = data['choices'][0]['message']['content'].strip()
            content = content.replace('```json','').replace('```','').strip()
            parsed  = json.loads(content)
            for k in NULL_SIGNALS.keys():
                parsed[k] = int(np.clip(parsed.get(k, 5), 0, 10))
            return parsed
        except json.JSONDecodeError:
            match = re.search(r'\{.*\}', content, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group())
                except:
                    pass
            if attempt < retries - 1:
                time.sleep(2**attempt)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2**attempt)
    return None

LLM_CACHE = f'{DATA_DIR}/llm_signals.csv'
if os.path.exists(LLM_CACHE):
    os.remove(LLM_CACHE)
    print('Deleted old LLM cache')

ratio_df = pd.read_csv(f'{DATA_DIR}/ratio_dataset.csv')
ratio_df['cik'] = ratio_df['cik'].astype(str).str.zfill(10)

llm_results = []
no_text     = 0
failed      = 0

print(f'Processing {len(ratio_df)} companies...')
print(f'Estimated time: ~25-30 minutes')

for i, row in ratio_df.iterrows():
    cik  = str(row['cik']).zfill(10)
    year = int(row['target_year'])
    text = get_item7_text(cik, year)
    if text is None:
        no_text += 1
        llm_results.append({'cik': cik, 'llm_status': 'no_text', **NULL_SIGNALS})
        continue
    token_count = len(enc.encode(text))
    signals = call_openrouter(text)
    if signals is None:
        failed += 1
        llm_results.append({'cik': cik, 'llm_status': 'failed', **NULL_SIGNALS})
    else:
        llm_results.append({'cik': cik, 'llm_status': 'ok', **signals})
    if (i+1) % 20 == 0:
        ok = sum(1 for r in llm_results if r.get('llm_status')=='ok')
        print(f'  [{i+1}/{len(ratio_df)}] ok={ok} no_text={no_text} failed={failed}')
        pd.DataFrame(llm_results).to_csv(LLM_CACHE, index=False)
    time.sleep(2.0)

llm_df = pd.DataFrame(llm_results)
llm_df['cik'] = llm_df['cik'].astype(str).str.zfill(10)
llm_df.to_csv(LLM_CACHE, index=False)

print(f'Done. OK={(llm_df["llm_status"]=="ok").sum()} No_text={no_text} Failed={failed}')
score_cols = list(NULL_SIGNALS.keys())
merged     = ratio_df.merge(llm_df, on='cik', how='left')
print(f'Scores by default status:')
print(merged.groupby('defaulted')[score_cols].mean().round(2))
print(f'Score distributions:')
print(llm_df[llm_df['llm_status']=='ok'][score_cols].describe().round(2))

Processing 166 companies...
Estimated time: ~25-30 minutes
  [60/166] ok=41 no_text=19 failed=0
  [80/166] ok=55 no_text=25 failed=0
  [100/166] ok=69 no_text=31 failed=0
  [140/166] ok=91 no_text=49 failed=0
  [160/166] ok=103 no_text=57 failed=0
Done. OK=105 No_text=61 Failed=0
Scores by default status:
           going_concern_severity  debt_risk_score  liquidity_risk_score  \
defaulted                                                                  
0                            2.38             2.59                  2.66   
1                            3.68             3.79                  4.33   

           management_distress  restructuring_risk  revenue_distress  
defaulted                                                             
0                         3.31                3.03              2.91  
1                         4.89                4.03              5.00  
Score distributions:
       going_concern_severity  debt_risk_score  liquidity_risk_score  \
count      

In [ ]:
# ── Cell 8: Merge Master Dataset ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import json

ratio_df   = pd.read_csv(f'{DATA_DIR}/ratio_dataset.csv')
finbert_df = pd.read_csv(f'{DATA_DIR}/finbert_signals.csv')
llm_df     = pd.read_csv(f'{DATA_DIR}/llm_signals.csv')

for df in [ratio_df, finbert_df, llm_df]:
    df['cik'] = df['cik'].astype(str).str.zfill(10)

print(f'Ratio  : {len(ratio_df)}')
print(f'FinBERT: {len(finbert_df)} | OK: {(finbert_df["finbert_status"]=="ok").sum()}')
print(f'LLM    : {len(llm_df)}    | OK: {(llm_df["llm_status"]=="ok").sum()}')

master = ratio_df.merge(finbert_df, on='cik', how='left')
master = master.merge(llm_df,      on='cik', how='left')
print(f'After merge: {master.shape}')

# ── FinBERT derived feature (Item 1A only) ────────────────────────────────────
# Net negativity of risk factors — primary FinBERT credit signal
master['finbert_1a_net_negative'] = (
    master['1a_negative'] - master['1a_positive']
).fillna(0)

# ── LLM scores — fill missing with median ────────────────────────────────────
LLM_SCORE_COLS = ['going_concern_severity','debt_risk_score','liquidity_risk_score',
                  'management_distress','restructuring_risk','revenue_distress']
for col in LLM_SCORE_COLS:
    median_val = llm_df[llm_df['llm_status']=='ok'][col].median()
    master[col] = master[col].fillna(median_val)
    print(f'  {col:<28} median fill = {median_val:.1f}')

# ── Feature groups ────────────────────────────────────────────────────────────
RATIO_FEATURES = [
    'leverage_ratio','debt_to_equity','debt_to_assets','current_ratio',
    'working_capital','roa','net_profit_margin','asset_turnover',
    'interest_coverage','ocf_to_debt','ocf_to_assets','ocf_margin','log_assets'
]
ZSCORE_FEATURES = ['altman_z','ohlson_o','ohlson_prob','z_distress','z_safe']
FINBERT_FEATURES = ['finbert_1a_net_negative']
LLM_FEATURES = LLM_SCORE_COLS
ALL_FEATURES = RATIO_FEATURES + ZSCORE_FEATURES + FINBERT_FEATURES + LLM_FEATURES
TARGET       = 'defaulted'

sparse = master[RATIO_FEATURES].isnull().sum(axis=1) > len(RATIO_FEATURES) * 0.6
master = master[~sparse].copy()

print(f'Final: {master.shape}')
print(f'Defaulted={master["defaulted"].sum()} Healthy={(master["defaulted"]==0).sum()}')
print(f'Default rate: {master["defaulted"].mean():.1%}')
print(f'Features: Ratio={len(RATIO_FEATURES)} ZScore={len(ZSCORE_FEATURES)} FinBERT={len(FINBERT_FEATURES)} LLM={len(LLM_FEATURES)} Total={len(ALL_FEATURES)}')

print(f'Correlation with default label (top 15):')
corr = master[ALL_FEATURES + ['defaulted']].corr()['defaulted'].drop('defaulted')
print(corr.sort_values(ascending=False).head(15).round(4))

master.to_csv(f'{DATA_DIR}/master_dataset.csv', index=False)
with open(f'{DATA_DIR}/feature_config.json','w') as f:
    json.dump({
        'ratio_features':   RATIO_FEATURES,
        'zscore_features':  ZSCORE_FEATURES,
        'finbert_features': FINBERT_FEATURES,
        'llm_features':     LLM_FEATURES,
        'all_features':     ALL_FEATURES,
        'target':           TARGET
    }, f, indent=2)
print('Saved master_dataset.csv + feature_config.json')

Ratio  : 166
FinBERT: 166 | OK: 105
LLM    : 166    | OK: 105
After merge: (166, 39)
  going_concern_severity       median fill = 3.0
  debt_risk_score              median fill = 3.0
  liquidity_risk_score         median fill = 4.0
  management_distress          median fill = 4.0
  restructuring_risk           median fill = 4.0
  revenue_distress             median fill = 4.0
Final: (166, 40)
Defaulted=77 Healthy=89
Default rate: 46.4%
Features: Ratio=13 ZScore=5 FinBERT=1 LLM=6 Total=25
Correlation with default label (top 15):
ohlson_o                   0.5945
debt_to_assets             0.4994
going_concern_severity     0.4068
management_distress        0.3945
debt_risk_score            0.3708
revenue_distress           0.3670
leverage_ratio             0.3622
liquidity_risk_score       0.3048
ohlson_prob                0.2022
finbert_1a_net_negative    0.1664
z_distress                 0.1627
restructuring_risk         0.1286
debt_to_equity             0.1028
asset_turnover          

In [ ]:
# ── Cell 9: PD + LGD + EL Modeling with Calibration ──────────────────────────
import lightgbm as lgb
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

master = pd.read_csv(f'{DATA_DIR}/master_dataset.csv')
with open(f'{DATA_DIR}/feature_config.json') as f:
    config = json.load(f)

ALL_FEATURES = config['all_features']
TARGET       = config['target']
X = master[ALL_FEATURES]
y = master[TARGET]

print(f'Dataset: {master.shape} | Features: {len(ALL_FEATURES)} | Default rate: {y.mean():.1%}')

# ── Time-based split — regulatory standard ────────────────────────────────────
train_mask = master['target_year'] <= 2017
test_mask  = master['target_year'] >= 2018
X_train = X[train_mask]; y_train = y[train_mask]
X_test  = X[test_mask];  y_test  = y[test_mask]
print(f'Time-based split:')
print(f'  Train (<=2017): {train_mask.sum()} | Default rate: {y_train.mean():.1%}')
print(f'  Test  (>=2018): {test_mask.sum()}  | Default rate: {y_test.mean():.1%}')
if y_test.nunique() < 2:
    print('WARNING: fallback to stratified split')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

n_neg, n_pos = (y_train==0).sum(), (y_train==1).sum()
print(f'Class balance: {n_neg} healthy | {n_pos} defaulted')

# ── Logistic Regression baseline ──────────────────────────────────────────────
lr_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1, random_state=42))
])
lr_pipe.fit(X_train, y_train)
lr_probs = lr_pipe.predict_proba(X_test)[:,1]
print(f'LR  AUC={roc_auc_score(y_test,lr_probs):.4f} AP={average_precision_score(y_test,lr_probs):.4f} Brier={brier_score_loss(y_test,lr_probs):.4f}')

# ── Calibrated LightGBM ───────────────────────────────────────────────────────
lgb_calibrated = CalibratedClassifierCV(
    lgb.LGBMClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=3, num_leaves=7,
        min_child_samples=15, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=1.0, reg_lambda=2.0, scale_pos_weight=n_neg/n_pos,
        random_state=42, verbose=-1
    ),
    method='isotonic', cv=5
)
lgb_calibrated.fit(X_train, y_train)
cal_probs = lgb_calibrated.predict_proba(X_test)[:,1]
print(f'LGB AUC={roc_auc_score(y_test,cal_probs):.4f} AP={average_precision_score(y_test,cal_probs):.4f} Brier={brier_score_loss(y_test,cal_probs):.4f} LogLoss={log_loss(y_test,cal_probs):.4f}')
print(f'PD dist (test): mean={cal_probs.mean():.3f} std={cal_probs.std():.3f} min={cal_probs.min():.3f} max={cal_probs.max():.3f}')

# ── 5-Fold CV ─────────────────────────────────────────────────────────────────
cv_model = CalibratedClassifierCV(
    lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, num_leaves=7,
                       min_child_samples=15, reg_alpha=1.0, reg_lambda=2.0,
                       scale_pos_weight=n_neg/n_pos, random_state=42, verbose=-1),
    method='isotonic', cv=3
)
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(cv_model, X, y, cv=cv,
                        scoring=['roc_auc','average_precision','neg_brier_score','neg_log_loss'])
aucs    = cv_res['test_roc_auc']
aps     = cv_res['test_average_precision']
briers  = -cv_res['test_neg_brier_score']
llosses = -cv_res['test_neg_log_loss']
print(f'5-Fold CV: AUC={aucs.mean():.4f}+-{aucs.std():.4f} | AP={aps.mean():.4f} | Brier={briers.mean():.4f} | LogLoss={llosses.mean():.4f}')

# Feature importance
base_est = lgb_calibrated.calibrated_classifiers_[0].estimator
imp_df   = pd.DataFrame({'feature':ALL_FEATURES,'importance':base_est.feature_importances_}).sort_values('importance',ascending=False)
print(f'Top 10 features:\n{imp_df.head(10).to_string(index=False)}')
print(f'Feature group contributions:')
for gname, gfeats in [('Ratio',config['ratio_features']),('Z-Score',config['zscore_features']),
                       ('FinBERT',config['finbert_features']),('LLM',config['llm_features'])]:
    gimp = imp_df[imp_df['feature'].isin(gfeats)]['importance'].sum()
    pct  = gimp / imp_df['importance'].sum() * 100
    print(f'  {gname:<12}: {gimp:>5.0f} ({pct:.1f}%)')

# Calibration curve
if y_test.nunique() == 2 and len(cal_probs) >= 10:
    fig, ax = plt.subplots(figsize=(7,6))
    prob_true, prob_pred = calibration_curve(y_test, cal_probs, n_bins=6)
    ax.plot(prob_pred, prob_true, 's-', color='#2563EB', label='Calibrated LGB')
    ax.plot([0,1],[0,1], 'k--', label='Perfect')
    ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
    ax.set_title('PD Calibration Curve'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'{DATA_DIR}/calibration_curve.png', dpi=150); plt.close()
    print('Calibration curve saved.')

# Refit on full data
lgb_final = CalibratedClassifierCV(
    lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, num_leaves=7,
                       min_child_samples=15, reg_alpha=1.0, reg_lambda=2.0,
                       scale_pos_weight=(y==0).sum()/(y==1).sum(), random_state=42, verbose=-1),
    method='isotonic', cv=5
)
lgb_final.fit(X, y)
master['PD'] = lgb_final.predict_proba(X)[:,1]

# LGD Model
def compute_lgd_proxy(row):
    base = 0.60
    if pd.notna(row['leverage_ratio']): base += (row['leverage_ratio'] - 0.5) * 0.20
    if pd.notna(row['current_ratio']):  base -= min(row['current_ratio'] - 1.0, 1.0) * 0.10
    if pd.notna(row['roa']):            base -= row['roa'] * 0.15
    return float(np.clip(base, 0.10, 0.95))

def_df        = master[master['defaulted']==1].copy()
def_df['LGD'] = def_df.apply(compute_lgd_proxy, axis=1)
LGD_FEATURES  = ['leverage_ratio','debt_to_assets','current_ratio','roa','log_assets',
                  'debt_risk_score','liquidity_risk_score']
lgd_X = def_df[LGD_FEATURES].fillna(def_df[LGD_FEATURES].median())
lgd_y = def_df['LGD']
lgd_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
lgd_model.fit(lgd_X, lgd_y)

full_lgd_X    = master[LGD_FEATURES].fillna(master[LGD_FEATURES].median())
master['LGD'] = np.clip(lgd_model.predict(full_lgd_X), 0.10, 0.95)
master['EAD'] = master['log_assets'].apply(np.expm1)
master['EL']  = master['PD'] * master['LGD'] * master['EAD']
master['EL_rate'] = master['PD'] * master['LGD']

master.to_csv(f'{DATA_DIR}/master_dataset.csv', index=False)
joblib.dump(lgb_final, f'{DATA_DIR}/lgb_pd_model.pkl')
joblib.dump(lgd_model, f'{DATA_DIR}/lgd_model.pkl')
joblib.dump(lr_pipe,   f'{DATA_DIR}/lr_pd_model.pkl')
print(f'PD dist (full): mean={master["PD"].mean():.3f} std={master["PD"].std():.3f}')
print(master[['name','defaulted','PD','LGD','EL_rate']].head(8).round(3).to_string())
print('All models saved.')

Dataset: (166, 40) | Features: 25 | Default rate: 46.4%
Time-based split:
  Train (<=2017): 130 | Default rate: 45.4%
  Test  (>=2018): 36  | Default rate: 50.0%
Class balance: 71 healthy | 59 defaulted
LR  AUC=0.9414 AP=0.9509 Brier=0.1074
LGB AUC=0.9552 AP=0.9508 Brier=0.0827 LogLoss=0.2714
PD dist (test): mean=0.481 std=0.414 min=0.000 max=1.000
5-Fold CV: AUC=0.9833+-0.0105 | AP=0.9791 | Brier=0.0600 | LogLoss=0.1952
Top 10 features:
                feature  importance
             log_assets          54
        working_capital          38
          ocf_to_assets          34
 going_concern_severity          30
            ocf_to_debt          24
         debt_to_assets          21
finbert_1a_net_negative          21
                    roa          14
      net_profit_margin          10
         leverage_ratio           8
Feature group contributions:
  Ratio       :   226 (75.8%)
  Z-Score     :    15 (5.0%)
  FinBERT     :    21 (7.0%)
  LLM         :    36 (12.1%)
Calibration cur

In [ ]:
# ── Cell 10: SHAP Explainability ──────────────────────────────────────────────
import shap, joblib

master = pd.read_csv(f'{DATA_DIR}/master_dataset.csv')
with open(f'{DATA_DIR}/feature_config.json') as f:
    config = json.load(f)
ALL_FEATURES = config['all_features']
lgb_final    = joblib.load(f'{DATA_DIR}/lgb_pd_model.pkl')
X            = master[ALL_FEATURES].copy()
base_lgb     = lgb_final.calibrated_classifiers_[0].estimator

explainer   = shap.TreeExplainer(base_lgb)
shap_values = explainer.shap_values(X)
sv          = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_df = pd.DataFrame(sv, columns=ALL_FEATURES)
shap_df['cik']       = master['cik'].values
shap_df['name']      = master['name'].values
shap_df['defaulted'] = master['defaulted'].values
shap_df['PD']        = master['PD'].values

mean_shap = shap_df[ALL_FEATURES].abs().mean().sort_values(ascending=False)
print('=== Global SHAP Importance ===')
print(mean_shap.head(15).round(4))

print('\n=== SHAP by feature group ===')
for gname, gfeats in [('Ratio',config['ratio_features']),('Z-Score',config['zscore_features']),
                       ('FinBERT',config['finbert_features']),('LLM',config['llm_features'])]:
    gshap = mean_shap[mean_shap.index.isin(gfeats)].sum()
    pct   = gshap / mean_shap.sum() * 100
    print(f'  {gname:<12}: {gshap:.4f} ({pct:.1f}%)')

print('\n=== Company Explanations ===')
high_idx = master[master['defaulted']==1]['PD'].idxmax()
low_idx  = master[master['defaulted']==0]['PD'].idxmin()
for idx, label in [(high_idx,'HIGH RISK'),(low_idx,'LOW RISK')]:
    name     = master.loc[idx,'name']
    pd_score = master.loc[idx,'PD']
    cshap    = shap_df.loc[idx, ALL_FEATURES].sort_values(key=abs, ascending=False)
    print(f'\n  [{label}] {name} | PD={pd_score:.3f}')
    for feat, val in cshap.head(8).items():
        direction = '↑ increases PD' if val > 0 else '↓ decreases PD'
        print(f'    {feat:<25}{val:>+8.4f}  {direction}')

shap_df.to_csv(f'{DATA_DIR}/shap_values.csv', index=False)
mean_shap.to_csv(f'{DATA_DIR}/shap_importance.csv', header=True)
print('\nSaved shap_values.csv + shap_importance.csv')

=== Global SHAP Importance ===
working_capital            0.7044
roa                        0.5850
log_assets                 0.5655
ocf_to_assets              0.4541
going_concern_severity     0.4334
finbert_1a_net_negative    0.3243
debt_to_assets             0.3224
ocf_to_debt                0.2967
ohlson_o                   0.0563
debt_risk_score            0.0542
ohlson_prob                0.0158
liquidity_risk_score       0.0143
debt_to_equity             0.0117
leverage_ratio             0.0072
ocf_margin                 0.0067
dtype: float64

=== SHAP by feature group ===
  Ratio       : 2.9601 (76.6%)
  Z-Score     : 0.0785 (2.0%)
  FinBERT     : 0.3243 (8.4%)
  LLM         : 0.5019 (13.0%)

=== Company Explanations ===

  [HIGH RISK] J. C. Penney Company, Inc. | PD=1.000
    working_capital           -1.0314  ↓ decreases PD
    roa                       +0.8029  ↑ increases PD
    debt_to_assets            +0.6240  ↑ increases PD
    ocf_to_assets             +0.6199  ↑ incre

In [ ]:
# ── Cell 11: Stress Test + Industry Concentration ─────────────────────────────
import joblib

master    = pd.read_csv(f'{DATA_DIR}/master_dataset.csv')
lgb_final = joblib.load(f'{DATA_DIR}/lgb_pd_model.pkl')
with open(f'{DATA_DIR}/feature_config.json') as f:
    config = json.load(f)
ALL_FEATURES = config['all_features']

base_X  = master[ALL_FEATURES].copy()
base_pd = lgb_final.predict_proba(base_X)[:,1]
base_el = (base_pd * master['LGD']).mean()

scenarios = {
    'Base Case': {},
    'Rate Shock (+200bps)': {
        'interest_coverage': lambda x: x - 1.5,
        'leverage_ratio':    lambda x: x + 0.10,
        'debt_risk_score':   lambda x: (x + 1.5).clip(0, 10),
        'liquidity_risk_score': lambda x: (x + 1.0).clip(0, 10),
    },
    'Recession (GDP -3%)': {
        'roa':                   lambda x: x - 0.03,
        'net_profit_margin':     lambda x: x - 0.05,
        'finbert_1a_net_negative': lambda x: (x + 0.10).clip(0, 1),
        'management_distress':   lambda x: (x + 1.5).clip(0, 10),
        'revenue_distress':      lambda x: (x + 2.0).clip(0, 10),
    },
    'Severe Stress (Both)': {
        'interest_coverage':     lambda x: x - 1.5,
        'leverage_ratio':        lambda x: x + 0.10,
        'roa':                   lambda x: x - 0.03,
        'net_profit_margin':     lambda x: x - 0.05,
        'finbert_1a_net_negative': lambda x: (x + 0.10).clip(0, 1),
        'debt_risk_score':       lambda x: (x + 1.5).clip(0, 10),
        'liquidity_risk_score':  lambda x: (x + 1.0).clip(0, 10),
        'management_distress':   lambda x: (x + 1.5).clip(0, 10),
        'revenue_distress':      lambda x: (x + 2.0).clip(0, 10),
    },
}

print('=== Stress Test ===')
print(f'{"Scenario":<35}{"Avg PD":>8}{"EL Rate":>10}{"ΔEL":>10}')
print('-'*65)
stress_rows = []
for name, shocks in scenarios.items():
    sx = base_X.copy()
    for feat, fn in shocks.items():
        if feat in sx.columns:
            sx[feat] = fn(sx[feat])
    sx['leverage_ratio']    = sx['leverage_ratio'].clip(0, 5)
    sx['interest_coverage'] = sx['interest_coverage'].clip(-50, 50)
    spd = lgb_final.predict_proba(sx)[:,1]
    sel = (spd * master['LGD']).mean()
    d   = (sel - base_el) / base_el * 100
    stress_rows.append({'scenario':name,'avg_pd':spd.mean(),'avg_el':sel,'delta_pct':d})
    print(f'  {name:<33}{spd.mean():>8.3f}{sel:>10.4f}{d:>+9.1f}%')

master['PD_base']     = base_pd
master['PD_recession']= lgb_final.predict_proba(
    base_X.assign(
        roa=base_X['roa']-0.03, net_profit_margin=base_X['net_profit_margin']-0.05,
        finbert_1a_net_negative=(base_X['finbert_1a_net_negative']+0.10).clip(0,1),
        management_distress=(base_X['management_distress']+1.5).clip(0,10),
        revenue_distress=(base_X['revenue_distress']+2.0).clip(0,10),
    )
)[:,1]

print('\n=== Industry Concentration (min 3 companies) ===')
ind = master.groupby('industry').agg(
    companies=('cik','count'), defaulted=('defaulted','sum'),
    avg_pd=('PD_base','mean'), avg_pd_rec=('PD_recession','mean'),
    avg_el_rate=('EL_rate','mean'),
).round(3)
ind['pd_delta%'] = ((ind['avg_pd_rec']-ind['avg_pd'])/ind['avg_pd'].replace(0,np.nan)*100).round(1)
ind = ind[ind['companies']>=3].sort_values('avg_el_rate', ascending=False)
print(ind.head(15).to_string())

pd.DataFrame(stress_rows).to_csv(f'{DATA_DIR}/stress_results.csv', index=False)
ind.to_csv(f'{DATA_DIR}/industry_concentration.csv')
master.to_csv(f'{DATA_DIR}/master_dataset.csv', index=False)
print('All results saved.')

=== Stress Test ===
Scenario                             Avg PD   EL Rate       ΔEL
-----------------------------------------------------------------
  Base Case                           0.462    0.3053     +0.0%
  Rate Shock (+200bps)                0.471    0.3107     +1.8%
  Recession (GDP -3%)                 0.517    0.3388    +11.0%
  Severe Stress (Both)                0.540    0.3529    +15.6%

=== Industry Concentration (min 3 companies) ===
                                 companies  defaulted  avg_pd  avg_pd_rec  avg_el_rate  pd_delta%
industry                                                                                         
Crude Petroleum and Natural Gas         17         17   0.977       0.987        0.671        1.0
Consumer Discretionary                   9          0   0.031       0.064        0.028      106.5
Information Technology                  10          0   0.044       0.084        0.027       90.9
Real Estate                              5          0 

In [ ]:
# ── Cell 12: Dashboard (Matplotlib) ──────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import json
import joblib

master   = pd.read_csv(f'{DATA_DIR}/master_dataset.csv')
stress   = pd.read_csv(f'{DATA_DIR}/stress_results.csv')
shap_imp = pd.read_csv(f'{DATA_DIR}/shap_importance.csv')
shap_imp.columns = ['feature','shap_importance']
shap_df  = pd.read_csv(f'{DATA_DIR}/shap_values.csv')

with open(f'{DATA_DIR}/feature_config.json') as f:
    config = json.load(f)

ALL_FEATURES = config['all_features']

def pd_to_grade(p):
    if p<0.10:   return 'AAA/AA'
    elif p<0.20: return 'A'
    elif p<0.35: return 'BBB'
    elif p<0.50: return 'BB'
    elif p<0.65: return 'B'
    elif p<0.80: return 'CCC'
    else:        return 'D'

master['grade'] = master['PD'].apply(pd_to_grade)
GRADE_ORDER     = ['AAA/AA','A','BBB','BB','B','CCC','D']

RED  = '#DC2626'
BLUE = '#2563EB'

fig, axes = plt.subplots(4, 2, figsize=(18, 28))
fig.suptitle('Credit Risk Grading System — Portfolio Analytics Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.subplots_adjust(hspace=0.45, wspace=0.35)

# ── Plot 1: PD Distribution ───────────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(master[master['defaulted']==1]['PD'], bins=20, alpha=0.7, color=RED,   label='Defaulted')
ax.hist(master[master['defaulted']==0]['PD'], bins=20, alpha=0.7, color=BLUE,  label='Healthy')
ax.set_title('PD Distribution by Default Status')
ax.set_xlabel('Probability of Default')
ax.set_ylabel('Company Count')
ax.legend()
ax.grid(alpha=0.3)

# ── Plot 2: SHAP Feature Importance ──────────────────────────────────────────
ax = axes[0, 1]
top12 = shap_imp.sort_values('shap_importance').tail(12)

def feat_color(f):
    if f in config['finbert_features']: return '#7C3AED'
    if f in config['llm_features']:     return '#0891B2'
    if f in config['zscore_features']:  return '#D97706'
    return BLUE

colors = [feat_color(f) for f in top12['feature']]
bars   = ax.barh(top12['feature'], top12['shap_importance'], color=colors)
ax.set_title('SHAP — Global Feature Importance')
ax.set_xlabel('Mean |SHAP|')
ax.grid(alpha=0.3, axis='x')

from matplotlib.patches import Patch
legend_elements = [
    Patch(color=BLUE,     label='Ratio'),
    Patch(color='#D97706',label='Z-Score'),
    Patch(color='#7C3AED',label='FinBERT'),
    Patch(color='#0891B2',label='LLM'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

# ── Plot 3: Stress Test ───────────────────────────────────────────────────────
ax = axes[1, 0]
stress_colors = ['#16A34A','#D97706','#D97706','#DC2626']
bars = ax.bar(stress['scenario'], stress['avg_el'], color=stress_colors)
for bar, delta in zip(bars, stress['delta_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{delta:+.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Stress Test — Portfolio EL Impact')
ax.set_ylabel('EL Rate')
ax.set_xticklabels(stress['scenario'], rotation=15, ha='right', fontsize=8)
ax.grid(alpha=0.3, axis='y')

# ── Plot 4: PD vs LGD Scatter ─────────────────────────────────────────────────
ax = axes[1, 1]
for label, color, name in [(1, RED, 'Defaulted'), (0, BLUE, 'Healthy')]:
    sub = master[master['defaulted']==label]
    sizes = sub['log_assets'].fillna(10) * 5
    ax.scatter(sub['PD'], sub['LGD'], c=color, alpha=0.6, s=sizes, label=name)
ax.set_title('PD vs LGD (sized by log assets)')
ax.set_xlabel('PD Score')
ax.set_ylabel('LGD')
ax.legend()
ax.grid(alpha=0.3)

# ── Plot 5: Credit Grade Distribution ────────────────────────────────────────
ax = axes[2, 0]
grade_counts = master['grade'].value_counts().reindex(GRADE_ORDER, fill_value=0)
grade_colors = ['#16A34A','#16A34A','#D97706','#D97706','#D97706','#DC2626','#DC2626']
bars = ax.bar(grade_counts.index, grade_counts.values, color=grade_colors)
for bar, val in zip(bars, grade_counts.values):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(val), ha='center', va='bottom', fontsize=9)
ax.set_title('Credit Grade Distribution')
ax.set_xlabel('Credit Grade')
ax.set_ylabel('Count')
ax.grid(alpha=0.3, axis='y')

# ── Plot 6: EL Rate Top 40 ────────────────────────────────────────────────────
ax = axes[2, 1]
top40 = master.nlargest(40, 'EL_rate')[['name','EL_rate','defaulted']].copy()
top40['short_name'] = top40['name'].str[:22]
bar_colors = [RED if d==1 else BLUE for d in top40['defaulted']]
ax.barh(top40['short_name'], top40['EL_rate'], color=bar_colors)
ax.set_title('EL Rate by Company (Top 40)')
ax.set_xlabel('EL Rate')
ax.tick_params(axis='y', labelsize=6)
ax.grid(alpha=0.3, axis='x')

# ── Plot 7: LLM Score Comparison ─────────────────────────────────────────────
ax = axes[3, 0]
llm_cols   = ['going_concern_severity','debt_risk_score','liquidity_risk_score',
               'management_distress','restructuring_risk','revenue_distress']
llm_labels = ['Going\nConcern','Debt\nRisk','Liquidity','Mgmt\nDistress','Restruc','Revenue\nDistress']
x = np.arange(len(llm_cols))
w = 0.35
d_means = [master[master['defaulted']==1][c].mean() for c in llm_cols]
h_means = [master[master['defaulted']==0][c].mean() for c in llm_cols]
ax.bar(x - w/2, d_means, w, color=RED,  alpha=0.8, label='Defaulted')
ax.bar(x + w/2, h_means, w, color=BLUE, alpha=0.8, label='Healthy')
ax.set_xticks(x)
ax.set_xticklabels(llm_labels, fontsize=8)
ax.set_title('LLM Scores — Defaulted vs Healthy')
ax.set_ylabel('Mean Score')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# ── Plot 8: SHAP Company Comparison ──────────────────────────────────────────
ax = axes[3, 1]
top8_feats = shap_imp.sort_values('shap_importance', ascending=False).head(8)['feature'].tolist()
high_idx   = master[master['defaulted']==1]['PD'].idxmax()
low_idx    = master[master['defaulted']==0]['PD'].idxmin()
high_shap  = shap_df.loc[high_idx, top8_feats] if high_idx in shap_df.index else pd.Series(0, index=top8_feats)
low_shap   = shap_df.loc[low_idx,  top8_feats] if low_idx  in shap_df.index else pd.Series(0, index=top8_feats)
high_name  = master.loc[high_idx,'name'][:20]
low_name   = master.loc[low_idx, 'name'][:20]

x          = np.arange(len(top8_feats))
ax.bar(x - w/2, high_shap.values, w, color=RED,  alpha=0.8, label=f'High: {high_name}')
ax.bar(x + w/2, low_shap.values,  w, color=BLUE, alpha=0.8, label=f'Low: {low_name}')
ax.set_xticks(x)
ax.set_xticklabels([f[:12] for f in top8_feats], rotation=30, ha='right', fontsize=7)
ax.set_title('SHAP — High Risk vs Low Risk')
ax.set_ylabel('SHAP Value')
ax.axhline(0, color='gray', linewidth=0.8)
ax.legend(fontsize=7)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/dashboard.png', dpi=150, bbox_inches='tight')
print('Dashboard saved.')
plt.show()

Dashboard saved.
